In [1]:
import requests
import json

### URL to use to search via search engine

In [2]:
# INDEX_PAGE = "https://idr-testing.openmicroscopy.org/webclient/?experimenter=-1"
INDEX_PAGE = "https://134.36.7.77/webclient/?experimenter=-1"
# SEARCH_ENGINE_URL = "https://idr-testing.openmicroscopy.org/searchengine/api/v1/resources/{type}/"
SEARCH_ENGINE_URL = "https://134.36.7.77/searchengine/api/v1/resources/{type}/"
# KEY_VALUE_SEARCH = SEARCH_ENGINE_URL + "search/?key={key}&value={value}"
KEY_VALUE_SEARCH = SEARCH_ENGINE_URL + "search/?key={key}&value={value}"
# KEYS_SEARCH = SEARCH_ENGINE_URL + "searchvaluesusingkey/?key={key}"
KEYS_SEARCH = SEARCH_ENGINE_URL + "searchvaluesusingkey/?key={key}"

In [3]:
INDEX_PAGE_NEW = "https://134.36.7.77/webclient/?experimenter=-1"
SEARCH_ENGINE_URL_NEW = "https://134.36.7.77/searchengine/api/v1/resources/{type}/"
KEY_VALUE_SEARCH_NEW = SEARCH_ENGINE_URL_NEW + "search/?key={key}&value={value}"
KEYS_SEARCH_NEW = SEARCH_ENGINE_URL_NEW + "searchvaluesusingkey/?key={key}"

### URLs to use to search via ``mapr``

In [4]:
# URL to use mapr
# MAPR_URL = "https://idr-testing.openmicroscopy.org/mapr/api/{key}/?value={value}&case_sensitive=false&orphaned=true"
MAPR_URL = "https://134.36.7.77/mapr/api/{key}/?value={value}&case_sensitive=false&orphaned=true"

# SCREENS_PROJECTS_URL = "https://idr-testing.openmicroscopy.org/mapr/api/{key}/?value={value}"
SCREENS_PROJECTS_URL = "https://134.36.7.77/mapr/api/{key}/?value={value}"

# PLATES_URL = "https://idr-testing.openmicroscopy.org/mapr/api/{key}/plates/?value={value}&id={screen_id}"
PLATES_URL = "https://134.36.7.77/mapr/api/{key}/plates/?value={value}&id={screen_id}"

# DATASETS_URL = "https://idr-testing.openmicroscopy.org/mapr/api/{key}/datasets/?value={value}&id={project_id}"
DATASETS_URL = "https://134.36.7.77/mapr/api/{key}/datasets/?value={value}&id={project_id}"

# IMAGES_URL = "https://idr-testing.openmicroscopy.org/mapr/api/{key}/images/?value={value}&node={parent_type}&id={parent_id}"
IMAGES_URL = "https://134.36.7.77/mapr/api/{key}/images/?value={value}&node={parent_type}&id={parent_id}"


In [5]:
# URL to use mapr
MAPR_URL_NEW = "https://134.36.7.77/mapr/api/{key}/?value={value}&case_sensitive=false&orphaned=true"
SCREENS_PROJECTS_URL_NEW = "https://134.36.7.77/mapr/api/{key}/?value={value}"
PLATES_URL_NEW = "https://134.36.7.77/mapr/api/{key}/plates/?value={value}&id={screen_id}"
DATASETS_URL_NEW = "https://134.36.7.77/mapr/api/{key}/datasets/?value={value}&id={project_id}"
IMAGES_URL_NEW = "https://134.36.7.77/mapr/api/{key}/images/?value={value}&node={parent_type}&id={parent_id}"


In [6]:
# create http session
with requests.Session() as session:
    request = requests.Request('GET', INDEX_PAGE)
    prepped = session.prepare_request(request)
    response = session.send(prepped)
    if response.status_code != 200:
        response.raise_for_status()

SSLError: HTTPSConnectionPool(host='134.36.7.77', port=443): Max retries exceeded with url: /webclient/?experimenter=-1 (Caused by SSLError(SSLCertVerificationError(1, "[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: IP address mismatch, certificate is not valid for '134.36.7.77'. (_ssl.c:1129)")))

In [7]:
# Key used by search engine
KEY = "Protein"
# Mapr equivalent key
KEY_MAPR = "protein"

### Load all the values for a specific key.
Only non empty value will be considered.

In [8]:
# Helper method to load the possible values for a given key
def load_values_for_given_key():
    values = []
    qs1 = {'type': 'image', 'key': KEY}
    url = KEYS_SEARCH.format(**qs1)  
    #json = session.get(url).json()
    json = session.get(url, verify=False).json()   
    for d in json['data']:
        if d['Value']:
            values.append(d['Value'])
    return values

In [9]:
values = load_values_for_given_key()

/Users/pwalczysko/opt/anaconda3/envs/idr_env/lib/python3.9/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '134.36.7.77'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [10]:
values.sort()

In [11]:
values

['bik',
 'calr-kdel',
 'cco',
 'cytb5',
 'don1',
 'dtmd-vamp1',
 'emerin',
 'ergic53',
 'galt',
 'golgin84',
 'heh1',
 'hsp104',
 'hta1',
 'htb1',
 'lamina',
 'lamp-1',
 'mao',
 'membrin',
 'ndc1',
 'neuromodulin',
 'nsr1',
 'nup1',
 'nup120',
 'nup159',
 'nup170',
 'nup188',
 'nup2',
 'nup49',
 'nup53',
 'nup57',
 'nup60',
 'nup82',
 'nup84',
 'pom34',
 'ptdss1',
 'pts-1',
 'rab11',
 'rab3',
 'rab5',
 'rab7',
 'ramp4',
 'spo20',
 'tetr',
 'vamp2',
 'vamp5',
 'vph1']

### Helper method to retrieve images using the search engine

In [12]:
# Helper method retrieving the result using directly the search api
def load_using_search_api(values):
    results = {}
    for item in values:
        ids = []
        qs1 = {'type': 'image', 'key': KEY, 'value': item}
        url = KEY_VALUE_SEARCH.format(**qs1)  
        json = session.get(url, verify=False).json()
        if 'results' in json['results']:
            images = json['results']['results']
            for image in images:
                if image['id'] not in ids:
                    ids.append(image['id'])
        results[item.lower()] = ids
    return results

In [13]:
# Helper method retrieving the result using directly the search api
def load_using_search_api_new(values):
    results = {}
    for item in values:
        ids = []
        qs1 = {'type': 'image', 'key': KEY, 'value': item}
        url = KEY_VALUE_SEARCH_NEW.format(**qs1)  
        json = session.get(url, verify=False).json()
        if 'results' in json['results']:
            images = json['results']['results']
            for image in images:
                if image['id'] not in ids:
                    ids.append(image['id'])
        results[item.lower()] = ids
    print(url)
    return results

### Helper method to retrieve images using ``mapr``

In [14]:
def get_items(values):
    items = []
    not_found = []
    for item in values:
        qs1 = {'key': KEY_MAPR, 'value': item}
        url = MAPR_URL.format(**qs1)
        json = session.get(url, verify=False).json()
        if len(json['maps']) == 0:
            not_found.append(item)
        for m in json['maps']: 
            items.append(m['id'])
    return items, not_found

def parse_annotation(images, json_data, item, name, data_type):
    screen_name = "-"
    plate_name = "-"
    project_name = "-"
    dataset_name = "-"
    if data_type == 'datasets':
        project_name = name
    else:
        screen_name = name
     
    for p in json_data[data_type]:
        parent_id = p['id']
        if data_type == 'datasets':
            dataset_name = p['name']
        else:
            plate_name = p['name']
        qs3 = {'key': KEY_MAPR, 'value': item,
                'parent_type': data_type[:-1], 'parent_id': parent_id}
        url3 = IMAGES_URL.format(**qs3)
        json = session.get(url3, verify=False).json()
        for i in json['images']:
            if i['id'] not in images:
                images.append(i['id'])
                                
def load_using_mapr(values):
    results = {}
    items, not_found = get_items(values)
    images = []
    for item in items:
        qs1 = {'key': KEY_MAPR, 'value': item}
        url1 = MAPR_URL.format(**qs1)
        json = session.get(url1, verify=False).json()
        for m in json['maps']:
            qs2 = {'key': KEY_MAPR, 'value': item}
            url2 = SCREENS_PROJECTS_URL.format(**qs2)
            json = session.get(url2, verify=False).json()
            for s in json['screens']:
                item = s['extra']['value']
                qs3 = {'key': KEY_MAPR, 'value': item, 'screen_id': s['id']}
                url3 = PLATES_URL.format(**qs3)
                parse_annotation(images, session.get(url3, verify=False).json(), item, s['name'], 'plates')
            for p in json['projects']:
                item = p['extra']['value']
                qs3 = {'key': KEY_MAPR, 'value': item, 'project_id': p['id']}
                url3 = DATASETS_URL.format(**qs3)
                parse_annotation(images, session.get(url3, verify=False).json(), item, p['name'], 'datasets')
        results[item.lower()] = images
    for n in not_found:
        results[n.lower()] = []
    return results
    

In [15]:
def get_items_new(values):
    items = []
    not_found = []
    for item in values:
        qs1 = {'key': KEY_MAPR, 'value': item}
        url = MAPR_URL_NEW.format(**qs1)
        json = session.get(url, verify=False).json()
        if len(json['maps']) == 0:
            not_found.append(item)
        for m in json['maps']: 
            items.append(m['id'])
    return items, not_found

def parse_annotation_new(images, json_data, item, name, data_type):
    screen_name = "-"
    plate_name = "-"
    project_name = "-"
    dataset_name = "-"
    if data_type == 'datasets':
        project_name = name
    else:
        screen_name = name
     
    for p in json_data[data_type]:
        parent_id = p['id']
        if data_type == 'datasets':
            dataset_name = p['name']
        else:
            plate_name = p['name']
        qs3 = {'key': KEY_MAPR, 'value': item,
                'parent_type': data_type[:-1], 'parent_id': parent_id}
        url3 = IMAGES_URL_NEW.format(**qs3)
        json = session.get(url3, verify=False).json()
        for i in json['images']:
            if i['id'] not in images:
                images.append(i['id'])
        print(url3)    
            
            
def load_using_mapr_new(values):
    results = {}
    items, not_found = get_items_new(values)
    images = []
    for item in items:
        qs1 = {'key': KEY_MAPR, 'value': item}
        url1 = MAPR_URL_NEW.format(**qs1)
        json = session.get(url1, verify=False).json()
        for m in json['maps']:
            qs2 = {'key': KEY_MAPR, 'value': item}
            url2 = SCREENS_PROJECTS_URL_NEW.format(**qs2)
            json = session.get(url2, verify=False).json()
            for s in json['screens']:
                item = s['extra']['value']
                qs3 = {'key': KEY_MAPR, 'value': item, 'screen_id': s['id']}
                url3 = PLATES_URL_NEW.format(**qs3)
                parse_annotation_new(images, session.get(url3, verify=False).json(), item, s['name'], 'plates')
            for p in json['projects']:
                item = p['extra']['value']
                qs3 = {'key': KEY_MAPR, 'value': item, 'project_id': p['id']}
                url3 = DATASETS_URL_NEW.format(**qs3)
                print(url3)
                parse_annotation_new(images, session.get(url3, verify=False).json(), item, p['name'], 'datasets')
        results[item.lower()] = images
    for n in not_found:
        results[n.lower()] = []
    return results
    

### Search using search engine 

In [16]:
# number of values to search for
s = 0
e = 500

In [17]:
%%time
results = load_using_search_api(values[s:e])

/Users/pwalczysko/opt/anaconda3/envs/idr_env/lib/python3.9/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '134.36.7.77'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/pwalczysko/opt/anaconda3/envs/idr_env/lib/python3.9/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '134.36.7.77'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/pwalczysko/opt/anaconda3/envs/idr_env/lib/python3.9/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '134.36.7.77'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnin

/Users/pwalczysko/opt/anaconda3/envs/idr_env/lib/python3.9/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '134.36.7.77'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/pwalczysko/opt/anaconda3/envs/idr_env/lib/python3.9/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '134.36.7.77'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/pwalczysko/opt/anaconda3/envs/idr_env/lib/python3.9/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '134.36.7.77'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnin

CPU times: user 1.08 s, sys: 115 ms, total: 1.2 s
Wall time: 3.89 s


/Users/pwalczysko/opt/anaconda3/envs/idr_env/lib/python3.9/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '134.36.7.77'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [18]:
%%time
results_new = load_using_search_api_new(values[s:e])

/Users/pwalczysko/opt/anaconda3/envs/idr_env/lib/python3.9/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '134.36.7.77'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/pwalczysko/opt/anaconda3/envs/idr_env/lib/python3.9/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '134.36.7.77'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/pwalczysko/opt/anaconda3/envs/idr_env/lib/python3.9/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '134.36.7.77'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnin

/Users/pwalczysko/opt/anaconda3/envs/idr_env/lib/python3.9/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '134.36.7.77'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/pwalczysko/opt/anaconda3/envs/idr_env/lib/python3.9/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '134.36.7.77'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/pwalczysko/opt/anaconda3/envs/idr_env/lib/python3.9/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '134.36.7.77'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnin

https://134.36.7.77/searchengine/api/v1/resources/image/search/?key=Protein&value=vph1
CPU times: user 1.03 s, sys: 92.8 ms, total: 1.13 s
Wall time: 3.69 s


/Users/pwalczysko/opt/anaconda3/envs/idr_env/lib/python3.9/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '134.36.7.77'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


### Search using ``mapr`` 

In [19]:
%%time
results_mapr = load_using_mapr(values[s:e])

/Users/pwalczysko/opt/anaconda3/envs/idr_env/lib/python3.9/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '134.36.7.77'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
%%time
results_mapr_new = load_using_mapr_new(values[s:e])

### Compare the outputs of the search

The checks below compare the keys e.g. gene list and the values i.e. image ids

In [ ]:
def dict_compare(d1, d2):
    d1_keys = set(d1.keys())
    d2_keys = set(d2.keys())
    shared_keys = d1_keys.intersection(d2_keys)
    added = d1_keys - d2_keys
    removed = d2_keys - d1_keys  
    modified = {o : (d1[o], d2[o]) for o in shared_keys if d1[o].sort() != d2[o].sort()}
    same = set(o for o in shared_keys if d1[o].sort() == d2[o].sort())
    return added, removed, modified, same

In [ ]:
added, removed, modified, same = dict_compare(results, results_new)

In [ ]:
assert len(added) == 0
assert len(removed) == 0
assert len(modified) == 0
assert len(same) == e - s

In [ ]:
len(results)

In [ ]:
results_new